In [1]:
# Cell 1: Path & imports
import sys
from pathlib import Path

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import torch
import dac

from core.rnndac_model import GRUModelConfig, RNNDACModel
from utils.io import load_checkpoint

DEVICE = "cpu"

/home/lonce/miniconda3/envs/synthformer/lib/python3.10/site-packages/audiotools/core/loudness.py:5: UserWarning: A NumPy version >=1.23.5 and <2.5.0 is required for this version of SciPy (detected version 1.22.3)
  import scipy


In [2]:
# Cell 2: Point to your checkpoint
# Edit this to match the run you want to load:
SAVENAME = "waterfill_test_postOC_6_heavycondweight"
RUN_TIMESTAMP = "20260630_120429"
STEP = 7500

ckpt_path = (
    Path.cwd() / "output" / f"{RUN_TIMESTAMP}_{SAVENAME}"
    / "checkpoints" / "checkpoints" / f"step_{STEP:06d}.pt"
)
print("Checkpoint:", ckpt_path)
assert ckpt_path.exists(), f"Not found: {ckpt_path}"

Checkpoint: /home/lonce/working/agent_projects/FargoNDacoder/dev_notebooks/output/20260630_120429_waterfill_test_postOC_6_heavycondweight/checkpoints/checkpoints/step_007500.pt


In [3]:
# Cell 3: Load checkpoint and reconstruct config
saved = load_checkpoint(str(ckpt_path), map_location=DEVICE)

model_config = GRUModelConfig(**saved["model_config"])
print(f"n_q={model_config.n_q}, hidden={model_config.hidden_size}, "
      f"cond_injection={model_config.cond_injection}")

n_q=9, hidden=128, cond_injection=concat


In [4]:
# Cell 4: Load DAC model
dac_model_path = dac.utils.download(model_type="44khz")

print(f"DAC weights: {dac_model_path}")
dac_model = dac.DAC.load(dac_model_path).to(DEVICE)
dac_model.eval();

DAC weights: /home/lonce/.cache/descript/dac/weights_44khz_8kbps_0.0.1.pth


/home/lonce/miniconda3/envs/synthformer/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


In [5]:
# Cell 5: Create RNN model and load trained weights
model = RNNDACModel(model_config, dac_model=dac_model).to(DEVICE)
model.load_state_dict(saved["model_state_dict"])
model.eval()
for p in model.parameters():
    p.requires_grad_(False)
print("Model loaded and frozen.")

Model loaded and frozen.


In [6]:
# Cell 6: Run inference off line to estimate RTF (Real Time Factor) 

from core.inference import infer_streaming_with_lookahead
frames=100
duration=frames*512/44100

params = {
    # Inference sampling params
    "infer_top_k": 5,
    "infer_temperature": 0.5,
}

cvect = torch.linspace(.05, .95, frames).unsqueeze(0).unsqueeze(-1)  # shape [1, 300, 1]
cvect=cvect.to(DEVICE)
print("cond_sequence device:", cvect.device)


import time
t0 = time.time()


with torch.no_grad():
    gen_audio = infer_streaming_with_lookahead(
        rnn_model=model,
        dac_model=dac_model,
        cond_sequence=cvect,   # [1, T, p]
        chunk_size=16,
        hop_size=8,
        right_context=4,
        frame_samples=512,
        tau=params["infer_temperature"],
        top_k=params["infer_top_k"],
        #pool=pool,  #Better with out pool selection!!
    )

print(f"Generated {duration:4f} seconds of sound in {time.time() - t0:.4f}s")
print(f"conditioning shape: {cvect.shape}")
print(f"Audio shape: {gen_audio.shape}")
print(f" T*512 = {cvect.shape[1]*512}")

cond_sequence device: cpu
Generated 1.160998 seconds of sound in 1.5809s
conditioning shape: torch.Size([1, 100, 1])
Audio shape: torch.Size([1, 1, 53248])
 T*512 = 51200


In [7]:
import IPython.display as ipd
gen_audio_np = gen_audio.squeeze().detach().cpu().numpy()
ipd.display(ipd.Audio(gen_audio_np, rate=44100))

In [8]:
# Cell 6: Launch real-time player
%gui tk
from rtplayer import SynthWidget

widget = SynthWidget(
    param_names=["fill level"],
    initial_values=[0.5],
    rnn_model=model,
    dac_model=dac_model,
    backend="aplay",
    verbose=False,    # suppress [slider] and [infer] prints (default)
)